# DIRAC: VISUAL module overview

*Gosia Olejniczak, Warsaw University, 2025*

The VISUAL module in DIRAC enables the calculation of various scalar, vector, and tensor fields representing unperturbed and perturbed densities.

This notebook demonstrates how to integrate and/or visualize these densities on custom grids for a few simple molecules. Note that we use a poor setup (e.g., small basis sets, coarse grids) for demonstration purposes in all examples, since all calculations run from scratch. For production calculations, the computational model needs to be well calibrated.

The calculations presented in this notebook can also be invoked by running the `visual_tutorial_overview` test (`ctest -R visual_tutorial_overview`) in the `DIRAC` automatic test system.

Here, we use the `py3Dmol` library for visualization and [RDKit](https://www.rdkit.org/) for molecular data manipulation, but any visualization software for plotting volumetric data can be used. We also demonstrate how to use different file formats, including HDF5, for which the `h5py` library is required. 
Please make sure that you have all required libraries and that DIRAC is successfully installed, and `pam` script is available (see the "Prerequisites" section below). 

### Prerequisites: environment preparation

If you are using `conda` package manager, then you can create a working environment as follows:

```
cd conda_envs
conda env create -f visual_overview.yml
cd ..
```

Make sure the environment is activated before running this notebook:

```
conda activate visual_overview
```

If successful, you should be able to execute the next cell without errors:

In [ ]:
import os,sys
from pathlib import Path
import shutil
import subprocess
from IPython.display import display, HTML

import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
Chem.Draw.IPythonConsole.ipython_3d = True
from rdkit import rdBase
from rdkit.Chem import rdmolfiles
from rdkit.Chem import rdDetermineBonds
from rdkit.Chem import rdMolAlign
from rdkit.Chem import rdMolTransforms

import py3Dmol

from ipywidgets import interact,fixed,IntSlider

Next, we set up paths to DIRAC `pam` script.

In this notebook, we assume the `pam` script is present in the `../build` directory. If you installed `DIRAC` in a different directory, adapt the paths as needed:

In [ ]:
# DIRAC
notebook_path=Path().parent.resolve()
dirac_path=notebook_path.parent.resolve()
dirac_bin=Path(dirac_path, 'build').resolve()
dirac_testset=Path(dirac_path, 'test').resolve()
pam=Path(dirac_bin, 'pam')

def run_dirac(cmd, log=False):
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if log:
        print(result.stdout, end="")

Print out the path to `pam` used in this notebook:

In [ ]:
cmd="which pam"
result=subprocess.run(cmd, shell=True, text=True, capture_output=True)
pam=result.stdout
print(pam)

Finally, we set up the working directory for all files used or generated in this notebook:

In [ ]:
test_name='visual_tutorial_overview'
workdir=Path(Path.cwd(), 'workdir', test_name).resolve()
workdir.mkdir(parents=True, exist_ok=True)
str(workdir)

#### Functions used in this tutorial

In [ ]:
def from_xyz_to_mol(fxyz):
    '''read xyz file, generate RDKit mol object'''
    raw_mol = rdmolfiles.MolFromXYZFile(str(fxyz))
    mol = Chem.Mol(raw_mol)
    rdDetermineBonds.DetermineConnectivity(mol)
    return mol

def view_single_mol(mol, add_volumetric=False, isovalue=0.1, cube_file=None):
    '''interactive visualization of mol (RDKit mol object)'''
    if add_volumetric:
        # mol + surface
        data = None
        with open(cube_file, "r") as infile:
            data = infile.read()
        view=py3Dmol.view(width=400, height=300)
        view.addVolumetricData(
            data,
            "cube",
            {
                "isoval": isovalue,
                "smoothness": 5,
                "opacity": 0.8,
                "volformat": "cube",
                "color": "blue",
            },
        )
        view.addModel(data, "cube")
        view.setStyle({"stick": {}, "sphere": {"scale": 0.3}})
        view.zoomTo()
    else:
        # only mol        
        view = py3Dmol.view(
            data=Chem.MolToMolBlock(mol),
            style={"stick": {}, "sphere": {"scale": 0.3}},
            width=300,
            height=300,
        )
    return view.show()

def dirac_output_file(dirname,finp,fmol):
    inp_string=str(finp).split('/')[-1].strip('.inp')
    mol_string=str(fmol).split('/')[-1].strip('.mol').strip('.xyz')       
    fout=Path(dirname, inp_string+'_'+mol_string+'.out').resolve()
    return str(fout)
    
def grep_from_dirac(pattern1, pattern2, file, exclude_last=False):
    if exclude_last:
        greplines="sed -n '/%s/,/^[^%s]/p' %s " %(pattern1, pattern2, file)
    else:
        greplines="sed -n '/%s/,/%s/p' %s " %(pattern1, pattern2, file)
    out=subprocess.run(greplines, shell=True, text=True, capture_output=True)
    print(out.stdout)

## Visualization of the unperturbed densities

In this section, we calculate various unperturbed densities of water molecule. First, we prepare input files in the working directory. We use input files from the 'visual_tutorial_overview' test in DIRAC automatic test set:

In [ ]:
os.chdir(str(workdir))
print('DIRAC will run in: %s' %(workdir))

In [ ]:
dirac_test=Path(dirac_testset, test_name).resolve()
fmol=str(Path(dirac_test, 'h2o.xyz').resolve())
finp1=str(Path(dirac_test, 'dc.inp').resolve())
finp2=str(Path(dirac_test, 'vis.inp').resolve())

print('The input files are in the corresponding DIRAC test directory: %s' %(dirac_test))

Let us first look at the molecule:

In [ ]:
mol=from_xyz_to_mol(fmol)

In [ ]:
view_single_mol(mol)

We are now prepared to run calculations. 

Note that in this tutorial, we run calculations in two steps, but they can also be done as one job. First, we optimize the wave function and save the result on the 'CHECKPOINT.h5' file, for which we use the `--outcmo` option. 

In [ ]:
cmd = 'pam --inp=%s --mol=%s --noarch --nobackup --outcmo' % (finp1, fmol)
print('DIRAC run command (step 1): ')
print(cmd)
run_dirac(cmd)

Then, we do calculations in the VISUAL module, using the generated 'CHECKPOINT.h5' file (the `--incmo` option). 

Note that DIRAC enables the calculation of many different densities on many different grids in the same input. We use this functionality and calculate all desired densities for our molecule at once. The essential block in the visualization input is:

In [ ]:
grep_from_dirac('VISUAL', 'END', finp2)

Here, we specified two grids (`.GRIDS` section of the input): the 3D rectilinear grid with 64x64x32 points with a 4.0 a.u. margin added around the molecule (grid `id=1`) and the grid used in the DFT calculations in step 1 (grid `id=2`).
In the following examples, we will use the first grid for visualization purposes and the second one for integration.

The `.GRIDFUNCTION` section informs about the densities to be calculated on these grids. We explore them in the following paragraphs. Now, we run these calculations (note it may take a few seconds to finish) by executing:

In [ ]:
cmd = 'pam --inp=%s --mol=%s --noarch --nobackup --incmo --get="*.cube"' % (finp2, fmol)
print('DIRAC run command (step 2): ')
print(cmd)
run_dirac(cmd)

### Visualization and integration of the electron density of water molecule

The first grid function specification line exports the electron density (`name=ed`) as a scalar field, which can be visualized (`purpose=visualization`). We export it in Gaussian `cube` format (`export=cube`) to the 'ed.cube' file (`file_out=ed.cube`). We visualize this density as a 3D surface at the isovalue=0.1:

In [ ]:
view_single_mol(mol, add_volumetric=True, isovalue=0.1, cube_file='ed.cube')

We also integrate the electron density (`purpose=integration`), as declared in the second grid function specification in the input. To inspect the result of integration, we look in the DIRAC output file and search for the "Final grid_function values" printout:

In [ ]:
fout=dirac_output_file(workdir,finp2,fmol)
grep_from_dirac('Final grid_function values', 'Elapsed', fout, exclude_last=True)

As expected, the integrated value is close to 10, which is the number of electrons in water molecule.

### Visualization of the electron localization function (ELF) of water molecule

In the third line of the visualization input, we asked to export the ELF (`name=elf`) as a scalar field, on the same grid used for the electron density visualization, to the `elf.cube` file. We visualize this density as a 3D surface at the isovalue=0.1:

In [ ]:
view_single_mol(mol, add_volumetric=True, isovalue=0.8, cube_file='elf.cube')

In [ ]:
# Clean working directory, move up:
Path(workdir, 'CHECKPOINT.h5').resolve().unlink(missing_ok=True)
os.chdir(str(notebook_path))

## Visualization of orbital densities

## Visualization of the magnetically-induced current density


## The magnetizability density


## The NMR shielding density


## Radial distributions
